# Práctica 6: Caso Práctico Integrador — Análisis de Tráfico Aéreo con OpenSky Network

**Informática · Grado en Ingeniería Aeroespacial · UCLM**  
Curso 2026/27

> **Instrucciones:** Guarda una copia de este notebook en tu Google Drive (**Archivo > Guardar una copia en Drive**) y resuelve los ejercicios en las celdas indicadas. Cada ejercicio cuenta con celdas para tu solución y celdas de comprobación para verificar su correcto funcionamiento.


## ✈️ Análisis de Tráfico Aéreo en Tiempo Real

En esta práctica integradora aplicaremos todas las estructuras vistas en la asignatura (condicionales, bucles, funciones, listas, diccionarios y conjuntos) sobre datos reales del sistema de vigilancia aeronáutica **ADS-B** (*Automatic Dependent Surveillance-Broadcast*), proporcionados por la red global de investigación **OpenSky Network**.

### Estructura del Vector de Estado (*State Vector*)
Cada aeronave detectada se describe mediante una lista con los siguientes campos normalizados:

| Índice | Campo | Tipo | Descripción |
|---|---|---|---|
| `0` | `icao24` | `str` | Identificador único hex de 24 bits del transponder ICAO |
| `1` | `callsign` | `str` | Indicativo de llamada del vuelo (ej. `"IBE3170"`, `"VLG2042"`) |
| `2` | `origin_country`| `str` | País de registro de la aeronave |
| `3` | `time_position` | `int` | Marca de tiempo UNIX de la última posición reportada |
| `4` | `last_contact`  | `int` | Marca de tiempo UNIX del último mensaje recibido |
| `5` | `longitude`     | `float` | Longitud geográfica en grados decimales |
| `6` | `latitude`      | `float` | Latitud geográfica en grados decimales |
| `7` | `baro_altitude` | `float` | Altitud barométrica en **metros** |
| `8` | `on_ground`     | `bool`  | `True` si la aeronave está en pista/rodadura, `False` en vuelo |
| `9` | `velocity`      | `float` | Velocidad horizontal respecto al suelo en **m/s** |
| `10`| `true_track`    | `float` | Rumbo real respecto al norte geográfico en grados ($[0, 360)$) |
| `11`| `vertical_rate` | `float` | Velocidad vertical en **m/s** (positivo = ascenso, negativo = descenso) |
| `12`| `sensors`       | `list`  | Identificadores de estaciones receptoras |
| `13`| `geo_altitude`  | `float` | Altitud geométrica GPS en metros |
| `14`| `squawk`        | `str`   | Código transponder de 4 dígitos asignado por ATC |
| `15`| `spi`           | `bool`  | Indicador de pulso especial de identificación (*Squawk Ident*) |
| `16`| `position_source`| `int`  | Origen del informe de posición |


### Datos de Telemetría de Muestra (Flota en Vuelo)


In [ ]:
# Dataset de prueba con vuelos representativos en el espacio aéreo europeo:
DATOS_OPENSKY = [
    ["34438b", "IBE3170 ", "Spain", 1718000000, 1718000005, -3.567, 40.489, 10668.0, False, 220.5, 45.0, 0.0, None, 10800.0, "1000", False, 0],
    ["345210", "VLG2042 ", "Spain", 1718000000, 1718000004, 2.078, 41.297, 8534.4, False, 205.0, 220.0, -8.5, None, 8600.0, "2140", False, 0],
    ["40062b", "BAW458  ", "United Kingdom", 1718000000, 1718000003, -0.461, 51.470, 11277.6, False, 235.0, 180.0, 0.0, None, 11400.0, "7700", False, 0],  # Emergencia general!
    ["3c6444", "DLH1102 ", "Germany", 1718000000, 1718000002, 8.570, 50.033, 9753.6, False, 215.0, 95.0, 5.2, None, 9850.0, "1000", False, 0],
    ["34210a", "AEA4011 ", "Spain", 1718000000, 1718000001, -3.700, 40.400, 0.0, True, 12.0, 310.0, 0.0, None, 600.0, "7000", False, 0],          # En tierra
    ["4ca123", "RYR301  ", "Ireland", 1718000000, 1718000005, -6.249, 53.421, 11887.2, False, 240.0, 135.0, 0.0, None, 12000.0, "7600", False, 0], # Fallo de radio!
    ["3465aa", "GES501  ", "Spain", 1718000000, 1718000004, -3.600, 40.500, None, False, 180.0, 45.0, None, None, None, "1000", False, 0],          # Incompleto (None)
    ["394a02", "AFR1420 ", "France", 1718000000, 1718000003, 2.550, 49.008, 10058.4, False, 225.0, 270.0, -2.1, None, 10200.0, "1000", False, 0]
]
print(f"Dataset cargado con {len(DATOS_OPENSKY)} vectores de estado.")


---
## Bloque 1: Limpieza de Datos y Conversión Aeronáutica

En los datos reales abundan registros incompletos (`None`) o aeronaves en tierra que no deben incluirse en el análisis de crucero.


### Ejercicio 1: Filtrado y limpieza de registros válidos

**Tareas:**
1. Define la función `limpiar_vuelos(flota_raw)`.
2. Filtra la lista para conservar **únicamente** las aeronaves que:
   * Tengan `on_ground == False` (estén en el aire).
   * Tengan valores no nulos (`is not None`) en `baro_altitude`, `velocity` y `true_track`.
3. Devuelve una nueva lista con los vuelos limpios y válidos.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    vuelos_limpios = limpiar_vuelos(DATOS_OPENSKY)
    print(f"Vuelos válidos en vuelo: {len(vuelos_limpios)} de {len(DATOS_OPENSKY)}")
    for v in vuelos_limpios:
        print(f" • Callsign: {v[1].strip():<8} | País: {v[2]:<15} | Altitud: {v[7]:>7.1f} m | Vel: {v[9]:>5.1f} m/s")
except NameError as e:
    print("Error:", e)


---
### Ejercicio 2: Conversor a unidades aeronáuticas estándar

La aviación comercial utiliza tradicionalmente nudos para la velocidad y pies para la altitud:
* $1\ \text{m/s} = 1.94384\ \text{nudos (kt)}$
* $1\ \text{metro} = 3.28084\ \text{pies (ft)}$
* **Nivel de Vuelo (*Flight Level*, FL):** $\text{FL} = \lfloor \text{altitud en pies} / 100 \rfloor$ (ej. $35\,000\ \text{ft} \rightarrow \text{FL350}$).

**Tareas:**
1. Define la función `convertir_unidades_aeronauticas(vector_vuelo)`.
2. La función recibe un vector de estado y devuelve un diccionario con:
   * `"callsign"`: Indicativo limpio sin espacios en blanco (`.strip()`).
   * `"velocidad_kt"`: Velocidad en nudos redondeada a 1 decimal.
   * `"velocidad_kmh"`: Velocidad en km/h ($v \cdot 3.6$) redondeada a 1 decimal.
   * `"altitud_ft"`: Altitud en pies redondeada a enteros.
   * `"flight_level"`: Cadena con el nivel de vuelo (ej. `"FL350"`).


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    if len(vuelos_limpios) > 0:
        info_aero = convertir_unidades_aeronauticas(vuelos_limpios[0])
        print("Informe aeronáutico del primer vuelo:", info_aero)
except NameError as e:
    print("Error:", e)


---
## Bloque 2: Lógica de Navegación Aérea


### Ejercicio 3: Clasificación de rumbo cardinal

El rumbo (`true_track`) se mide en grados sexagesimales de $0^\circ$ a $360^\circ$.

Clasificación en 8 sectores cardinales:
* **Norte (N):** $[337.5^\circ, 360^\circ) \cup [0^\circ, 22.5^\circ)$
* **Noreste (NE):** $[22.5^\circ, 67.5^\circ)$
* **Este (E):** $[67.5^\circ, 112.5^\circ)$
* **Sureste (SE):** $[112.5^\circ, 157.5^\circ)$
* **Sur (S):** $[157.5^\circ, 202.5^\circ)$
* **Suroeste (SW):** $[202.5^\circ, 247.5^\circ)$
* **Oeste (W):** $[247.5^\circ, 292.5^\circ)$
* **Noroeste (NW):** $[292.5^\circ, 337.5^\circ)$

**Tareas:**
1. Define la función `obtener_rumbo_cardinal(true_track)`.
2. Devuelve la etiqueta de rumbo correspondiente (`"N"`, `"NE"`, `"E"`, etc.).


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    test_rumbos = [0, 45, 90, 180, 220, 270, 315, 350]
    for r in test_rumbos:
        print(f"Rumbo {r:>3}° -> {obtener_rumbo_cardinal(r)}")
except NameError as e:
    print("Error:", e)


---
### Ejercicio 4: Clasificación de fase vertical de vuelo

La velocidad vertical (`vertical_rate`) indica si el avión está maniobrando en altitud:
* **Ascenso:** `vertical_rate > 1.0` $\text{m/s}$
* **Descenso:** `vertical_rate < -1.0` $\text{m/s}$
* **Crucero nivelado:** $-1.0 \le \text{vertical\_rate} \le 1.0$ $\text{m/s}$

**Tareas:**
1. Define la función `determinar_fase_vertical(vertical_rate)`.
2. Devuelve `"Ascenso"`, `"Descenso"` o `"Crucero"`.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    print("+5.2 m/s ->", determinar_fase_vertical(5.2))
    print("-8.5 m/s ->", determinar_fase_vertical(-8.5))
    print(" 0.0 m/s ->", determinar_fase_vertical(0.0))
except NameError as e:
    print("Error:", e)


---
## Bloque 3: Panel de Control y Alertas de Seguridad ATC


### Ejercicio 5: Detección de códigos transponder Squawk de emergencia

En control de tráfico aéreo (ATC), existen tres códigos Squawk internacionales reservados para emergencias críticas:
* `7700`: **Emergencia general** (fallo de motor, despresurización, etc.).
* `7600`: **Fallo total de comunicaciones de radio** (*Nordo*).
* `7500`: **Interferencia ilícita / Secuestro**.

**Tareas:**
1. Define la función `detectar_emergencias(flota)`.
2. Recorre la lista de vuelos e identifica aquellas aeronaves cuyo campo `squawk` coincida con uno de los tres códigos de emergencia.
3. Devuelve una lista de tuplas `(callsign, codigo_squawk, tipo_emergencia)`.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    alertas = detectar_emergencias(DATOS_OPENSKY)
    print(f"Alertas de emergencia activas: {len(alertas)}")
    for a in alertas:
        print(f"🚨 ALERTA: Vuelo {a[0].strip()} con código Squawk {a[1]} ({a[2]})")
except NameError as e:
    print("Error:", e)


---
### Ejercicio 6: Radar de proximidad y conflictos de separación aérea

Para evitar colisiones en vuelo, el sistema TCAS monitoriza la distancia entre pares de aeronaves.

Aproximación de distancia horizontal entre dos coordenadas $(\text{lat}_1, \text{lon}_1)$ y $(\text{lat}_2, \text{lon}_2)$ en km:
$$d_{\text{horiz}} \approx 111.0 \cdot \sqrt{(\text{lat}_1 - \text{lat}_2)^2 + ((\text{lon}_1 - \text{lon}_2) \cdot \cos(\text{lat}_{\text{media}}))^2}$$

Existe un **conflicto potencial de separación** si:
* Distancia horizontal $d_{\text{horiz}} < 20.0\ \text{km}$
* Separación vertical $\Delta h = |h_1 - h_2| < 300.0\ \text{m}$ (aproximadamente $1\,000\ \text{ft}$)

**Tareas:**
1. Define la función `detectar_conflictos_proximidad(flota_limpia)`.
2. Compara todos los pares de aeronaves sin repetir pares ($i < j$).
3. Devuelve una lista de tuplas `(callsign_1, callsign_2, dist_horiz_km, sep_vert_m)` con las parejas en conflicto.


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    # Añadimos un vuelo cercano para probar el detector
    vuelo_cercano = ["349999", "IBE3171 ", "Spain", 1718000000, 1718000005, -3.550, 40.480, 10800.0, False, 220.0, 45.0, 0.0, None, 10800.0, "1000", False, 0]
    flota_test = vuelos_limpios + [vuelo_cercano]
    
    conflictos = detectar_conflictos_proximidad(flota_test)
    print(f"Conflictos detectados: {len(conflictos)}")
    for c in conflictos:
        print(f"⚠️ CONFLICTO DE SEPARACIÓN: {c[0].strip()} <-> {c[1].strip()} | Dist horiz: {c[2]:.2f} km | Sep vert: {c[3]:.1f} m")
except NameError as e:
    print("Error:", e)


---
## Bloque 4: Estadísticas de Flota con Diccionarios y Sets


### Ejercicio 7: Distribución por país y aerolíneas activas

**Tareas:**
1. Define la función `estadisticas_flota(flota_limpia)`.
2. Calcula y retorna un diccionario con:
   * `"conteo_paises"`: Diccionario con el número de aeronaves registradas por país (`origin_country`).
   * `"paises_unicos"`: Conjunto (`set`) con todos los países sin duplicados.
   * `"aerolineas_prefijos"`: Conjunto (`set`) con los prefijos OACI de 3 letras de los callsigns en vuelo (ej. `"IBE"`, `"VLG"`, `"BAW"`).


In [ ]:
# Tu solución aquí


In [ ]:
# Celda de comprobación:
try:
    stats = estadisticas_flota(vuelos_limpios)
    print("Aeronaves por país:", stats["conteo_paises"])
    print("Países presentes:  ", stats["paises_unicos"])
    print("Aerolíneas (OACI): ", stats["aerolineas_prefijos"])
except NameError as e:
    print("Error:", e)


---

**Departamento de Tecnologías y Sistemas de Información**  
Escuela de Ingeniería Industrial y Aeroespacial de Toledo — UCLM
